In [ ]:
# Setup
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

# GPU memory config
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

print(f"TensorFlow version: {tf.__version__}")
print("GPU Available:", tf.config.list_physical_devices('GPU'))

np.random.seed(42)
tf.random.set_seed(42)

# Constants
LATENT_DIM = 128
KL_WEIGHT = 16
BATCH_SIZE = 512
LEARNING_RATE = 0.0003
CLIPNORM = 2.0
EPOCHS = 100

BASE_FILTERS = 64
NUM_LAYERS = 4
BOTTLENECK_DIM = 4
# BOTTLENECK_DIM * 2(NUM_LAYERS + 1) MUST = IMAGE_SIZE (e.g. 4 * 2(5) = 64)
IMAGE_SIZE_X = 64
IMAGE_SIZE_Y = 64
CHANNELS = 3

2026-04-16 00:37:07.206981: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-16 00:37:07.883925: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-16 00:37:09.998578: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# Sampling Layer
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = keras.random.normal(shape=keras.ops.shape(z_mean), dtype=z_mean.dtype)
        return z_mean + keras.ops.exp(0.5 * z_log_var) * epsilon

# Encoder
def build_encoder(latent_dim=LATENT_DIM):
    inputs = keras.Input(shape=(IMAGE_SIZE_X, IMAGE_SIZE_Y, CHANNELS))
   
    x = inputs
    for i in range(NUM_LAYERS):
         filters = BASE_FILTERS * (2 ** i)
         x = layers.Conv2D(filters, 3, strides=2, padding='same', activation='relu')(x)
         x = layers.BatchNormalization()(x)
    
    x = layers.Flatten()(x)
    x = layers.Dropout(0.2)(x)
    
    z_mean    = layers.Dense(latent_dim, name='z_mean')(x)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
    z         = Sampling()([z_mean, z_log_var])
    
    return keras.Model(inputs, [z_mean, z_log_var, z], name='encoder')

# Decoder
def build_decoder(latent_dim=LATENT_DIM):
    inputs = keras.Input(shape=(latent_dim,))

    final_filters = BASE_FILTERS * (2 ** (NUM_LAYERS - 1))
    
    x = layers.Dense(BOTTLENECK_DIM * BOTTLENECK_DIM * final_filters, activation='relu')(inputs)
    x = layers.Reshape((BOTTLENECK_DIM, BOTTLENECK_DIM, final_filters))(x)

    for i in range(NUM_LAYERS - 1, -1, -1):
        filters = BASE_FILTERS * (2 ** i)
        x = layers.Conv2DTranspose(filters, 3, strides=2, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
    
    outputs = layers.Conv2D(CHANNELS, 3, padding='same', activation='tanh', dtype='float32')(x)
    
    return keras.Model(inputs, outputs, name='decoder')

# VAE
class VAE(keras.Model):
    def __init__(self, encoder, decoder, kl_weight, **kwargs):
        super().__init__(**kwargs)
        self.encoder   = encoder
        self.decoder   = decoder
        self.kl_weight = kl_weight

    def call(self, x):
        z_mean, z_log_var, z = self.encoder(x)
        return self.decoder(z)

    def _compute_losses(self, x, reconstruction, z_mean, z_log_var):
        image_size = tf.cast(tf.reduce_prod(tf.shape(x)[1:3]), tf.float32)
        recon_loss = tf.reduce_mean(
            keras.losses.mse(x, reconstruction)
        ) * image_size
        kl_loss = -0.5 * tf.reduce_mean(
            z_log_var - tf.square(z_mean) - tf.exp(z_log_var) + 1
        )
        total_loss = recon_loss + self.kl_weight * kl_loss
        return {"loss": total_loss, "recon_loss": recon_loss, "kl_loss": kl_loss}

    def train_step(self, data):
        x, _ = data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x, training=True)
            reconstruction       = self.decoder(z, training=True)
            losses               = self._compute_losses(x, reconstruction, z_mean, z_log_var)
        grads = tape.gradient(losses["loss"], self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        return losses

    def test_step(self, data):
        x, _ = data
        z_mean, z_log_var, z = self.encoder(x, training=False)
        reconstruction       = self.decoder(z, training=False)
        return self._compute_losses(x, reconstruction, z_mean, z_log_var)

encoder = build_encoder(latent_dim=LATENT_DIM)
decoder = build_decoder(latent_dim=LATENT_DIM)
vae     = VAE(encoder, decoder, kl_weight=KL_WEIGHT, name='vae')
vae.compile(optimizer=keras.optimizers.Adam(
    learning_rate=LEARNING_RATE,
    clipnorm=CLIPNORM
))
print("VAE built successfully")

I0000 00:00:1776296231.792413     857 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


VAE built successfully


In [ ]:
# Load CelebA

# Run in terminal to load CelebA dataset (if not already installed in environment): 
# pip install tensorflow-datasets
# pip install importlib-resources
# pip install bs4

import tensorflow_datasets as tfds

dataset, info = tfds.load('celeb_a', split=['train', 'validation', 'test'], with_info=True, as_supervised=False)
train_data = dataset[0]
val_data   = dataset[1]

# Preprocessing for training (w/ augmentation)
def preprocess_train(sample):
    image = tf.cast(sample['image'], tf.float32)
    image = tf.image.resize_with_crop_or_pad(image, 178, 178)
    image = tf.image.resize(image, [IMAGE_SIZE_X, IMAGE_SIZE_Y])
    image = tf.image.random_flip_left_right(image)
    image = (image / 127.5) - 1.0
    return image, image

# Preprocessing for validation (no augmentation)
def preprocess_val(sample):
    image = tf.cast(sample['image'], tf.float32)
    image = tf.image.resize_with_crop_or_pad(image, 178, 178)
    image = tf.image.resize(image, [IMAGE_SIZE_X, IMAGE_SIZE_Y])
    image = (image / 127.5) - 1.0
    return image, image

# tf.data pipeline
train_dataset = train_data.map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE).shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset   = val_data.map(preprocess_val, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Callbacks
checkpoint = keras.callbacks.ModelCheckpoint(
    'best_vae.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.75, patience=5, min_lr=1e-6
)

history = vae.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=[early_stopping, lr_scheduler]
)

/home/username/tf-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 100%|██████████| 5/5 [03:17<00:00, 39.55s/ url]


Dataset celeb_a downloaded and prepared to /home/username/tensorflow_datasets/celeb_a/2.1.0. Subsequent calls will reuse this data.
Epoch 1/100


2026-04-16 00:42:26.984730: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-04-16 00:42:26.984799: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-04-16 00:42:27.666511: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d7ad0034560 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-16 00:42:27.666550: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-04-16 00:42:28.841069: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-16 00:42:30.298810: I external/local_xla/xla/stream_executor/cuda/cuda

1271/1272 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - kl_loss: 1432.8192 - loss: 23178.8473 - recon_loss: 253.7410

...

1272/1272 ━━━━━━━━━━━━━━━━━━━━ 94s 73ms/step - kl_loss: 1.7618 - loss: 136.5604 - recon_loss: 108.3720 - val_kl_loss: 1.7572 - val_loss: 137.4862 - val_recon_loss: 109.3707 - learning_rate: 3.0000e-04
Epoch 13/100
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 95s 74ms/step - kl_loss: 1.7490 - loss: 134.1123 - recon_loss: 106.1289 - val_kl_loss: 1.7372 - val_loss: 136.5300 - val_recon_loss: 108.7345 - learning_rate: 3.0000e-04
Epoch 14/100
   2/1272 ━━━━━━━━━━━━━━━━━━━━ 1:30 71ms/step - kl_loss: 1.7502 - loss: 137.9753 - recon_loss: 109.9715  

2026-04-16 01:03:34.536688: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 12582912 bytes after encountering the first element of size 12582912 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


1272/1272 ━━━━━━━━━━━━━━━━━━━━ 95s 74ms/step - kl_loss: 1.7473 - loss: 132.0218 - recon_loss: 104.0644 - val_kl_loss: 1.7602 - val_loss: 134.4038 - val_recon_loss: 106.2399 - learning_rate: 3.0000e-04
Epoch 15/100
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 98s 76ms/step - kl_loss: 1.7651 - loss: 131.9463 - recon_loss: 103.7046 - val_kl_loss: 1.7629 - val_loss: 134.8970 - val_recon_loss: 106.6907 - learning_rate: 3.0000e-04
Epoch 16/100
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 98s 76ms/step - kl_loss: 1.7316 - loss: 140.4280 - recon_loss: 112.7222 - val_kl_loss: 1.7417 - val_loss: 133.8028 - val_recon_loss: 105.9362 - learning_rate: 3.0000e-04
Epoch 17/100
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 96s 75ms/step - kl_loss: 1.7427 - loss: 142.2903 - recon_loss: 114.4078 - val_kl_loss: 1.7433 - val_loss: 135.0541 - val_recon_loss: 107.1607 - learning_rate: 3.0000e-04
Epoch 18/100
   2/1272 ━━━━━━━━━━━━━━━━━━━━ 1:55 91ms/step - kl_loss: 1.7518 - loss: 125.2694 - recon_loss: 97.2399

2026-04-16 01:10:02.265052: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 12582912 bytes after encountering the first element of size 12582912 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


 552/1272 ━━━━━━━━━━━━━━━━━━━━ 31:05 3s/step - kl_loss: 1.7454 - loss: 131.1657 - recon_loss: 103.2398

In [ ]:
'''TO-DO: show reconstructions and generations every 5-10 epochs instead of just loss curves to visualise the learning process?'''

# Function to convert [-1, 1] images to [0, 1] for display
def tanh_display(img):
    return (np.clip(img, -1, 1) + 1) / 2

# Reconstruct 10 test images
test_samples  = x_test[:10]
reconstructed = vae.predict(test_samples, verbose=0)

# Show originals vs reconstructions
fig = plt.figure(figsize=(25, 5))
gs = fig.add_gridspec(4, 10, height_ratios=[0.15, 1, 0.15, 1], hspace=0.05)

# Label rows
for label, row in [('Original', 0), ('Reconstructed', 2)]:
    ax = fig.add_subplot(gs[row, :])
    ax.text(0.5, 0.5, label, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.axis('off')

# Image rows
for i in range(10):
    ax = fig.add_subplot(gs[1, i])
    ax.imshow(tanh_display(test_samples[i]))
    ax.axis('off')

    ax = fig.add_subplot(gs[3, i])
    ax.imshow(tanh_display(reconstructed[i]))
    ax.axis('off')

plt.suptitle('VAE Reconstructions', fontsize=16)
plt.tight_layout()
plt.show()

# Generate 10 new images
n_samples = 10
random_latent = np.random.normal(size=(n_samples, LATENT_DIM))
generated = decoder.predict(random_latent, verbose=0)

fig, axes = plt.subplots(1, 10, figsize=(20, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(tanh_display(generated[i]))
    ax.axis('off')

plt.suptitle('VAE Generations', fontsize=16)
plt.tight_layout()
plt.show()

# Training loss curve
plt.figure(figsize=(25, 5))
plt.plot(history.history['loss'],     label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True)
plt.show()

print(f"Final training loss:   {history.history['loss'][-1]:.4f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.4f}")